# Airbnb Paris – Enhanced (TabPFN-Embeddings)
- TabPFN wird **auf dem Label der Trainingszeilen** gefittet, `get_embeddings` liefert die Repräsentation
- Fit-Kontext auf <= 9000 Zeilen begrenzt (TabPFN-Sample-Limit)
- ⚠️ Label-Leak per Konstruktion → semi-supervised, nicht mit den unsupervised Varianten vergleichbar

In [ ]:
import os
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, classification_report, precision_recall_curve, auc
from tabpfn import TabPFNClassifier

SEED = int(os.environ.get("SEED", 1))
print("SEED", SEED)

## Daten & Split laden

In [ ]:
df = pd.read_csv(f"../../data/preprocessed/cleaned_airbnb_paris_seed{SEED}.csv")
split = pd.read_csv(f"../../data/splits/split_airbnb_paris_seed{SEED}.csv")
feat = [c for c in df.columns if c not in ["row_id", "is_top_rating"]]
X, y = df[feat], df["is_top_rating"]

tr = df["row_id"].isin(split.loc[split["split"] == "train", "row_id"]).values
va = df["row_id"].isin(split.loc[split["split"] == "val", "row_id"]).values
print("Train:", int(tr.sum()), "| Val:", int(va.sum()), "| Features:", len(feat))

## Fit-Kontext aus Train ziehen & TabPFN fitten
- Nur Trainingszeilen; bei > 9000 stratifiziert heruntergesamplet

In [ ]:
X_tr, y_tr = X[tr], y[tr]
if len(X_tr) > 9000:
    X_ctx, _, y_ctx, _ = train_test_split(X_tr, y_tr, train_size=9000, stratify=y_tr, random_state=SEED)
else:
    X_ctx, y_ctx = X_tr, y_tr

clf = TabPFNClassifier()
clf.fit(X_ctx.values, y_ctx.values)
print("Kontext:", len(X_ctx))

## Sanity-Check auf Val
- Zeigt, ob die Embeddings überhaupt Signal tragen; Test bleibt unberührt

In [ ]:
outlier_label = y_ctx.value_counts().idxmin()
outlier_col = sorted(np.unique(y_ctx).tolist()).index(outlier_label)
proba = clf.predict_proba(X[va].values)[:, outlier_col]
y_bin = (y[va] == outlier_label).astype(int)

prec, rec, _ = precision_recall_curve(y_bin, proba)
print("AUPRC  ", round(auc(rec, prec), 4))
print("AUC-ROC", round(roc_auc_score(y_bin, proba), 4))
print(classification_report(y[va], clf.predict(X[va].values), digits=4, zero_division=0))

## Embeddings extrahieren (Mittel über Estimatoren) & speichern

In [ ]:
emb = np.asarray(clf.get_embeddings(X.values))
if emb.ndim == 3:
    emb = emb.mean(axis=0)
print("Embedding-Dimension:", emb.shape[1])

out = pd.concat([df[["row_id"]].reset_index(drop=True),
                 pd.DataFrame(emb, columns=[f"emb_{i}" for i in range(emb.shape[1])])], axis=1)
out["is_top_rating"] = df["is_top_rating"].values
out.to_csv(f"../../data/preprocessed/enhanced_airbnb_paris_seed{SEED}.csv", index=False)
print("Shape:", out.shape)

## Verifikation

In [ ]:
assert out.isna().sum().sum() == 0
assert out["row_id"].is_unique